# 12. Biomarker Scoring Test

Pipeline step ⑩. Verifies `derive_biomarkers()` — the final integration step that
converts FeatureRecords and BiomechRecords into:

1. **BiomarkerRecord** — individual metric pass-throughs with provenance
2. **BiomarkerScoreRecord** — per-rep composite score (0–100) using Z-score deduction
   against the synthetic-normal baseline at `data/reference/baseline_zscore.json`

Scores represent deviation from a reference movement quality baseline,
not absolute clinical measurements.

Pipeline position: Biomech Proxy → **Biomarker Derivation**

This notebook assumes that the following notebooks are already passing:
- 00_environment_check through 11_biomechanical_proxy_test

Reference: `docs/pipeline/10_biomarker_scoring.md`,
`docs/terminology.md` (Synthetic-normal baseline, Movement quality score)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

from movement.annotation import apply_annotation, load_annotation_csv
from movement.biomech import extract_rep_biomech
from movement.biomarker import BiomarkerRecord, from_biomech_record, from_feature_record
from movement.biomarker.scoring import BiomarkerScoreRecord, derive_biomarkers
from movement.config import LANDMARKS, make_coordinate_columns, make_required_columns, make_visibility_columns
from movement.exercise_definition import load_exercise_definition
from movement.features import extract_rep_features
from movement.io import load_pose_csv
from movement.normalization import normalize_pose_by_hip_torso
from movement.pipeline import (
    AnnotationConfig, BiomechConfig, BiomarkerConfig, ExerciseDefinitionConfig,
    FeaturesConfig, MotionAttributionConfig, NormalizationConfig,
    PhaseSegmentationConfig, PipelineConfig, ValidationConfig, run_pipeline,
)
from movement.segmentation import segment_phases
from movement.validation import run_basic_validation

print('imports OK')

## Data Setup

Runs full pipeline ①–⑥ to produce feature + biomech records.

In [ ]:
csv_path = '../data/pose/sample/mediapipe_squat_synthetic.csv'
ann_path = '../data/pose/sample/mediapipe_squat_synthetic_annotation.csv'
def_dir  = '../data/definitions/exercises'

df_raw = load_pose_csv(csv_path)
run_basic_validation(
    df=df_raw,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)
ann_df       = load_annotation_csv(ann_path)
df_ann, _    = apply_annotation(df_raw, ann_df)
exercise_def = load_exercise_definition(exercise_id='squat', definitions_dir=def_dir)
df_norm, _   = normalize_pose_by_hip_torso(df=df_ann, landmarks=LANDMARKS)
df_seg, _    = segment_phases(df_norm, exercise_def, fps_default=30.0)

feat_records   = extract_rep_features(df_seg, exercise_def)
biomech_records = extract_rep_biomech(df_seg, exercise_def, use_visibility_weight=True)

print(f'feature records : {len(feat_records)}')
print(f'biomech records : {len(biomech_records)}')

## Direct derive_biomarkers() Test

In [ ]:
baseline_path = Path('../data/reference/baseline_zscore.json')

biomarker_records, score_records = derive_biomarkers(
    feat_records=feat_records,
    biomech_records=biomech_records,
    exercise_definition=exercise_def,
    definition_version=exercise_def.version,
    baseline_path=baseline_path,
)

print(f'BiomarkerRecord count    : {len(biomarker_records)}')
print(f'BiomarkerScoreRecord count: {len(score_records)}')

## Check 1: BiomarkerRecord Fields and irovenance

In [ ]:
assert len(biomarker_records) > 0, 'no BiomarkerRecords produced'
for r in biomarker_records:
    assert isinstance(r, BiomarkerRecord)
    assert r.biomarker_id,            f'biomarker_id empty'
    assert r.exercise_id == 'squat',  f'wrong exercise_id'
    assert r.value is not None
    assert len(r.source_fields) > 0,  f'source_fields empty for {r.biomarker_id}'
print(f'PASS: all {len(biomarker_records)} BiomarkerRecord fields valid')
print('Sample records:')
for r in biomarker_records[:4]:
    print(f'  {r.biomarker_id:45s}  rep={r.rep_id}  value={r.value:.4f}  unit={r.unit}')

## Check 2: BiomarkerScoreRecord — One ier Rep, Score 0–100

In [ ]:
assert len(score_records) > 0, 'no BiomarkerScoreRecords produced'
for s in score_records:
    assert isinstance(s, BiomarkerScoreRecord)
    assert 0.0 <= s.final_score <= 100.0, f'score out of range: {s.final_score} (rep {s.rep_id})'
    assert s.rep_id is not None
print(f'PASS: BiomarkerScoreRecord scores within 0–100')
for s in score_records:
    print(f'  rep={s.rep_id}  final_score={s.final_score:.2f}  '
          f'spatial={s.spatial_score:.2f}  temporal={s.temporal_score:.2f}  '
          f'control={s.control_score:.2f}  biomech={s.biomech_score:.2f}')

## Check 3: Domain Score Weights (spatial 40 %, temporal 30 %, control 20 %, biomech 10 %)

In [ ]:
for s in score_records:
    reconstructed = (s.spatial_score  * 0.40
                   + s.temporal_score * 0.30
                   + s.control_score  * 0.20
                   + s.biomech_score  * 0.10)
    diff = abs(s.final_score - reconstructed)
    assert diff < 1.0, (
        f'rep {s.rep_id}: final_score={s.final_score:.2f} '
        f'does not match weighted sum={reconstructed:.2f}'
    )
print('PASS: final_score = 0.40·spatial + 0.30·temporal + 0.20·control + 0.10·biomech')

## Check 4: Dynamic Floor > 0

In [ ]:
for s in score_records:
    assert s.dynamic_floor >= 0.0, f'dynamic_floor negative: {s.dynamic_floor}'
    assert s.final_score >= s.dynamic_floor, (
        f'rep {s.rep_id}: final_score {s.final_score:.2f} below dynamic_floor {s.dynamic_floor:.2f}'
    )
print('PASS: final_score >= dynamic_floor for all reps')
for s in score_records:
    print(f'  rep={s.rep_id}  dynamic_floor={s.dynamic_floor:.2f}')

## Check 5: as_dict() Serialization

In [ ]:
required_score_keys = [
    'rep_id', 'exercise_id', 'final_score', 'dynamic_floor',
    'spatial_score', 'temporal_score', 'control_score', 'biomech_score',
]
for s in score_records:
    d = s.as_dict()
    for k in required_score_keys:
        assert k in d, f'missing key {k} in BiomarkerScoreRecord.as_dict()'
print('PASS: as_dict() contains all required keys')
print(json.dumps(score_records[0].as_dict(), indent=2, default=str))

## Check 6: Pipeline Integration

In [ ]:
cfg = PipelineConfig()
cfg.validation          = ValidationConfig(enabled=True)
cfg.annotation          = AnnotationConfig(enabled=True, path=ann_path)
cfg.exercise_definition = ExerciseDefinitionConfig(enabled=True,
                              definitions_dir=def_dir, exercise_id='squat')
cfg.normalization       = NormalizationConfig(enabled=True)
cfg.phase_segmentation  = PhaseSegmentationConfig(enabled=True)
cfg.motion_attribution  = MotionAttributionConfig(enabled=True)
cfg.features            = FeaturesConfig(enabled=True)
cfg.biomech             = BiomechConfig(enabled=True)
cfg.biomarker           = BiomarkerConfig(enabled=True)

pipe_df, pipe_report = run_pipeline(df_raw, config=cfg, landmarks=LANDMARKS)

assert 'biomarker'        in pipe_report
assert 'biomarker_scores' in pipe_report
n_scores = len(pipe_report['biomarker_scores'])
print(f'PASS: pipeline ⑩ biomarker_scores: {n_scores} reps')
for s in pipe_report['biomarker_scores']:
    print(f'  rep={s["rep_id"]}  final_score={s["final_score"]:.2f}')
print(f'steps executed: {list(pipe_report.keys())}')

## Interpretation

| Score component | Weight | Biomechanical meaning |
|---|---|---|
| Spatial | 40 % | ROM, symmetry, trajectory shape |
| Temporal | 30 % | Tempo consistency, rep-to-rep variability |
| Control | 20 % | CoM stability, compensatory movements |
| Biomech | 10 % | CoM path, joint moment arm ratio |

**Score interpretation**: Scores close to 100 indicate that the movement matches
the synthetic-normal reference distribution on all domains. Deductions represent
deviation magnitude (Z-score) on individual biomarkers, not clinical severity.
The dynamic floor prevents the score from falling below the mandatory-ROM minimum.